In [3]:
# =========================================================
# CUSTOMER SUBSCRIPTION PREDICTION MODEL
# =========================================================

# -----------------------------
# IMPORT LIBRARIES
# -----------------------------
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer

from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

import joblib
import warnings
warnings.filterwarnings("ignore")

# -----------------------------
# LOAD DATASET
# -----------------------------
df = pd.read_csv("customer_shopping_behavior.csv")

# -----------------------------
# REMOVE USELESS COLUMN
# -----------------------------
df.drop("Customer ID", axis=1, inplace=True)

# -----------------------------
# HANDLE MISSING VALUES
# -----------------------------
df["Review Rating"].fillna(
    df["Review Rating"].mean(),
    inplace=True
)

# -----------------------------
# TARGET COLUMN
# -----------------------------
target = "Subscription Status"

# -----------------------------
# FEATURES & TARGET
# -----------------------------
X = df.drop(target, axis=1)

y = df[target]

# -----------------------------
# CATEGORICAL & NUMERICAL
# -----------------------------
categorical_cols = X.select_dtypes(include=["object"]).columns

numerical_cols = X.select_dtypes(exclude=["object"]).columns

# -----------------------------
# PREPROCESSING
# -----------------------------
num_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="mean"))
])

cat_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_transformer, numerical_cols),
        ("cat", cat_transformer, categorical_cols)
    ]
)

# -----------------------------
# MODEL PIPELINE
# -----------------------------
model = Pipeline(steps=[
    ("preprocessor", preprocessor),

    ("classifier", RandomForestClassifier(
        n_estimators=300,
        max_depth=15,
        random_state=42
    ))
])

# -----------------------------
# TRAIN TEST SPLIT
# -----------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

# -----------------------------
# TRAIN MODEL
# -----------------------------
print("Training Model...")

model.fit(X_train, y_train)

print("Training Completed!")

# -----------------------------
# PREDICTIONS
# -----------------------------
y_pred = model.predict(X_test)

# -----------------------------
# ACCURACY
# -----------------------------
accuracy = accuracy_score(y_test, y_pred)

print("\n================ RESULTS ================")

print(f"Accuracy : {accuracy * 100:.2f}%")

# -----------------------------
# CLASSIFICATION REPORT
# -----------------------------
print("\nClassification Report:\n")

print(classification_report(y_test, y_pred))

# -----------------------------
# CONFUSION MATRIX
# -----------------------------
print("\nConfusion Matrix:\n")

print(confusion_matrix(y_test, y_pred))

# -----------------------------
# SAMPLE PREDICTIONS
# -----------------------------
results = pd.DataFrame({
    "Actual": y_test.values,
    "Predicted": y_pred
})

print("\nSample Predictions:")

print(results.head(10))

# -----------------------------
# SAVE MODEL
# -----------------------------
joblib.dump(model, "subscription_prediction_model.pkl")

print("\nModel Saved Successfully!")

# =========================================================
# SINGLE PREDICTION
# =========================================================

sample = X.iloc[[0]]

prediction = model.predict(sample)

print("\nSingle Prediction:")

print(prediction)

# =========================================================
# END
# =========================================================

Training Model...
Training Completed!

================ RESULTS ================
Accuracy : 82.44%

Classification Report:

              precision    recall  f1-score   support

          No       0.99      0.76      0.86       558
         Yes       0.62      0.99      0.76       222

    accuracy                           0.82       780
   macro avg       0.81      0.87      0.81       780
weighted avg       0.89      0.82      0.83       780


Confusion Matrix:

[[424 134]
 [  3 219]]

Sample Predictions:
  Actual Predicted
0    Yes       Yes
1     No        No
2    Yes       Yes
3     No        No
4     No        No
5    Yes       Yes
6     No        No
7     No        No
8     No        No
9     No       Yes

Model Saved Successfully!

Single Prediction:
['Yes']


In [4]:
joblib.dump(model, "subscription_prediction_model.pkl")

print("\nPKL File Saved Successfully!")



PKL File Saved Successfully!
